# 2. Training CatBoost models

## Goal

Cross-validated phase classification and multi-output
transition-temperature regression, including out-of-fold predictions,
model persistence, feature importance and SHAP interpretation.

## Setup

The notebook opens the small prepared example but leaves the expensive
fits disabled. Set `RUN_CATBOOST_TRAINING = True` only after selecting a
sufficiently large modelling table, targets and an output directory.

### 1. Configure data, descriptors and run control


In [1]:
from pathlib import Path

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
EXAMPLE_DATA = PROJECT_ROOT / "examples" / "data"
PAPER_DATA = PROJECT_ROOT / "paper" / "data" / "processed"
MODEL_DIRECTORY = PROJECT_ROOT / "models"
RESULT_DIRECTORY = PROJECT_ROOT / "results"
RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True)

import pandas as pd

from lc_qspr import PHASE_TARGETS, TEMPERATURE_TARGETS
from lc_qspr.descriptors import select_descriptor_names
from lc_qspr.preprocessing import read_table, separate_chiral_phases

DATA_FILE = EXAMPLE_DATA / "sample_desc.csv"
RUN_CATBOOST_TRAINING = False
DESCRIPTOR_FAMILY = "All"
N_SPLITS = 5

table, chiral_phases = separate_chiral_phases(read_table(DATA_FILE))
feature_names = select_descriptor_names(table, DESCRIPTOR_FAMILY)
phase_targets = [target for target in PHASE_TARGETS if target in table]
temperature_targets = [target for target in TEMPERATURE_TARGETS if target in table]
print(table.shape, len(feature_names), len(phase_targets), len(temperature_targets))

(5, 1654) 1613 11 24


### 2. Create multi-label cross-validation folds

**Caution.** Every retained class needs enough examples for the requested number of folds; inspect target counts before fitting.

**Extension.** Use the saved split indices across model families when making paired performance comparisons.

In [2]:
from lc_qspr.evaluation import cross_validation_splits

splits = None
if RUN_CATBOOST_TRAINING:
    splits = cross_validation_splits(
        table, phase_targets,
        mode="classification", n_splits=N_SPLITS, random_state=42,
    )
    print("Fold sizes:", [(len(train), len(test)) for train, test in splits])
else:
    print("Training is disabled. Set RUN_CATBOOST_TRAINING to True after reviewing class counts.")

Training is disabled. Set RUN_CATBOOST_TRAINING to True after reviewing class counts.


### 3. Train the phase-existence classifier

**Caution.** The distributed model used 5,000 boosting rounds; this can take hours on the full descriptor database.

**Extension.** Adjust `CatBoostConfig` in one visible place and retain the out-of-fold table for an unbiased performance estimate.

In [3]:
from lc_qspr.catboost_training import CatBoostConfig, train_catboost_phases

phase_result = None
if RUN_CATBOOST_TRAINING:
    phase_result = train_catboost_phases(
        table, feature_names, phase_targets, splits,
        config=CatBoostConfig(iterations=5000, learning_rate=0.3),
        model_path=RESULT_DIRECTORY / "CatBoost_phase.cbm",
    )
    phase_result.predictions.to_csv(
        RESULT_DIRECTORY / "CatBoost_phase_oof.csv", index=True
    )
    pd.Series(feature_names, name="descriptor").to_csv(
        RESULT_DIRECTORY / "CatBoost_feature_names.csv", index=False
    )
    display(phase_result.summary)

### 4. Train the transition-temperature regressor

**Caution.** Missing temperatures are expected and are handled by `MultiRMSEWithMissingValues`; do not replace them with zero.

**Extension.** Separate single-target models can be studied by selecting one temperature column, although they no longer share information across transitions.

In [4]:
from lc_qspr.catboost_training import train_catboost_temperatures

temperature_result = None
if RUN_CATBOOST_TRAINING:
    temperature_result = train_catboost_temperatures(
        table, feature_names, temperature_targets, splits,
        config=CatBoostConfig(iterations=10000, learning_rate=0.3),
        model_path=RESULT_DIRECTORY / "CatBoost_temperature.cbm",
    )
    temperature_result.predictions.to_csv(
        RESULT_DIRECTORY / "CatBoost_temperature_oof.csv", index=True
    )
    display(temperature_result.summary)

### 5. Inspect feature importance

**Caution.** Built-in importance describes the fitted model, not a causal molecular mechanism, and correlated descriptors can divide importance.

**Extension.** Compare physical descriptor families or repeat the analysis across folds to assess stability.

In [5]:
from lc_qspr.catboost_training import catboost_feature_importance
from lc_qspr.plotting import plot_catboost_importance

if temperature_result is not None:
    importance = catboost_feature_importance(
        temperature_result.models[-1], feature_names, top_n=40
    )
    plot_catboost_importance(importance)

### 6. Calculate SHAP values

**Caution.** SHAP on thousands of molecules and descriptors is memory-intensive; start with a bounded validation sample.

**Extension.** Stratify the interpretation sample by phase or scaffold when comparing chemical subgroups.

In [6]:
from lc_qspr.catboost_training import catboost_shap_values

if temperature_result is not None:
    interpretation_sample = table.loc[:, feature_names].head(200)
    shap_values = catboost_shap_values(
        temperature_result.models[-1], interpretation_sample
    )
    import shap
    shap.summary_plot(
        shap_values,
        interpretation_sample,
        feature_names=feature_names,
    )
    print("SHAP output shape:", getattr(shap_values, "shape", "multi-output list"))